# Substantia nigra scANVi data processing

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.io import mmread

Download data from http://ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE178265 (https://www.nature.com/articles/s41593-022-01061-1)

In [ ]:
! wget https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE178265&format=file&file=GSE178265%5FHomo%5Fbcd%2Etsv%2Egz
! wget https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE178265&format=file&file=GSE178265%5FHomo%5Ffeatures%2Etsv%2Egz
! wget https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE178265&format=file&file=GSE178265%5FHomo%5Fmatrix%2Emtx%2Egz

Create adata object:

In [ ]:
adata_human= sc.read_mtx('data/GSE178265_Homo_matrix.mtx').T
bc_human= pd.read_csv('data/GSE178265_Homo_bcd.tsv', delimiter = None, header = None)
features_human = pd.read_csv('data/GSE178265_Homo_features.tsv', delimiter = '\t', header = None)
adata_human.obs_names = bc_human[0]
adata_human.var_names = features_human[0]
adata_human

In [ ]:
adata_human.obs.index = adata_human.obs.index.astype(str)
adata_human.var.index = adata_human.var.index.astype(str)
adata_human.write('out/human.h5ad')

Add metadata: downloaded from https://singlecell.broadinstitute.org/single_cell/study/SCP1768/ 

In [ ]:
meta = pd.read_csv('data/METADATA_PD.tsv', sep = '\t', header = [0,1])
human_meta = meta[meta['species__ontology_label']['group'] == 'Homo sapiens']

## cell type coordinates
astro_umap = pd.read_csv('data/astro_UMAP.tsv', sep = '\t', header = [0,1])
da_umap = pd.read_csv('data/da_UMAP.tsv', sep = '\t', header = [0,1])
endo_umap = pd.read_csv('data/endo_UMAP.tsv', sep = '\t', header = [0,1])
mg_umap = pd.read_csv('data/mg_UMAP.tsv', sep = '\t', header = [0,1])
nonda_umap = pd.read_csv('data/nonda_UMAP.tsv', sep = '\t', header = [0,1])
olig_umap = pd.read_csv('data/olig_UMAP.tsv', sep = '\t', header = [0,1])
opc_umap = pd.read_csv('data/opc_UMAP.tsv', sep = '\t', header = [0,1])

astrocytes_in_meta = list(set(human_meta['NAME']['TYPE'].tolist()) & set(astro_umap['NAME']['TYPE'].tolist()))
dopaminergic_neurons_in_meta = list(set(human_meta['NAME']['TYPE'].tolist()) & set(da_umap['NAME']['TYPE'].tolist()))
endothelial_in_meta = list(set(human_meta['NAME']['TYPE'].tolist()) & set(endo_umap['NAME']['TYPE'].tolist()))
microglia_in_meta = list(set(human_meta['NAME']['TYPE'].tolist()) & set(mg_umap['NAME']['TYPE'].tolist()))
non_dopaminergic_neurons_in_meta = list(set(human_meta['NAME']['TYPE'].tolist()) & set(nonda_umap['NAME']['TYPE'].tolist()))
oligodendrocytes_in_meta = list(set(human_meta['NAME']['TYPE'].tolist()) & set(olig_umap['NAME']['TYPE'].tolist()))
oligo_precursors_in_meta = list(set(human_meta['NAME']['TYPE'].tolist()) & set(opc_umap['NAME']['TYPE'].tolist()))

astrocytes = astro_umap['NAME']['TYPE'].tolist()
dopaminergic_neurons = da_umap['NAME']['TYPE'].tolist()
endothelial = endo_umap['NAME']['TYPE'].tolist()
microglia = mg_umap['NAME']['TYPE'].tolist()
non_dopaminergic_neurons = nonda_umap['NAME']['TYPE'].tolist()
oligodendrocytes = olig_umap['NAME']['TYPE'].tolist()
oligo_precursors = opc_umap['NAME']['TYPE'].tolist()

In [ ]:
## add FACS sorting data (NR4A2 positive or negative)
h = human_meta[['NAME', 'FACS_Classification']]
h.index = h['NAME']['TYPE']
pos = h[h['FACS_Classification']['group'] == 'Positive']
neg = h[h['FACS_Classification']['group'] == 'Negative']
pos_ind = pos.index.tolist()
neg_ind = neg.index.tolist()
adata_human.obs['NR4A2'] = 'Positive'
for val in adata_human.obs.index:
    if val in neg_ind:
        adata_human.obs['NR4A2'][val] = 'Negative'

In [ ]:
## add cell type data
cell_types = [astrocytes,dopaminergic_neurons, endothelial,microglia,non_dopaminergic_neurons,oligodendrocytes,oligo_precursors]
adata_human.obs['cell_type'] = 'NA'
for val in adata_human.obs.index:
    if val in astrocytes:
        adata_human.obs['cell_type'][val] = 'Astro'
    if val in dopaminergic_neurons:
        adata_human.obs['cell_type'][val] = 'DA'
    if val in endothelial:
        adata_human.obs['cell_type'][val] = 'Endo'
    if val in microglia:
        adata_human.obs['cell_type'][val] = 'Micro'
    if val in non_dopaminergic_neurons:
        adata_human.obs['cell_type'][val] = 'non_DA_neuron'
    if val in oligodendrocytes:
        adata_human.obs['cell_type'][val] = 'Oligo'
    if val in oligo_precursors:
        adata_human.obs['cell_type'][val] = 'OPC'

In [ ]:
## add subtype 
cell_types_dict = {}
umap_dict = {'astro': astro_umap,
             'da': da_umap,
             'endo': endo_umap,
             'mg': mg_umap,
             'nonda': nonda_umap,
             'olig': olig_umap,
             'opc': opc_umap}
for k in umap_dict.keys():
    print(k)
    print(umap_dict[k]['Cell_Type']['group'].value_counts())
    cell_types_dict[k] = umap_dict[k]['Cell_Type']['group'].unique().tolist()
    print()

cell_subtypes = {}
for key in cell_types_dict:
    for val in cell_types_dict[key]:
        cell_subtypes[val] = []

for key in umap_dict.keys():
    umap_dict[key].index = umap_dict[key]['NAME']['TYPE']

combined_df = pd.concat(umap_dict.values())
combined_df_short = pd.DataFrame(combined_df['Cell_Type']['group'])
df = pd.DataFrame(index = combined_df_short.index.tolist(), columns = ['subtype'])
df['subtype']=combined_df_short['group'].tolist()

for val in df.index:
    for j in cell_subtypes.keys():
        if df['subtype'][val] == j:
            cell_subtypes[j].append(val)

adata_human.obs['subtype'] = 'NA'
for k in cell_subtypes.keys():
    for val in cell_subtypes[k]:
        adata_human.obs['subtype'][val] = k

In [ ]:
## remove NA cells and add extra metadata
column_name = 'subtype'
adata_human_filtered = adata_human[adata_human.obs[column_name] != 'NA']
l = adata_human_filtered.obs.index.tolist()
human_meta.index = human_meta['NAME']['TYPE'].tolist()
human_meta.columns = ['NAME','libname','biosample_id','donor_id','species','species__ontology_label','disease','disease__ontology_label','organ','organ__ontology_label','library_preparation_protocol','library_preparation_protocol__ontology_label','sex','date','Donor_Age','Donor_PMI','Status','Cause_of_Death','FACS_Classification']
filtered_cells_meta = human_meta.loc[l]
adata_human_filtered.obs = pd.merge(adata_human_filtered.obs,filtered_cells_meta,how='outer',left_index=True,right_index=True)
adata_human_filtered.obs['donor_id'] = adata_human_filtered.obs['donor_id'].astype(str)

In [ ]:
## save 
adata_human_filtered.write('out/human_filtered_meta.h5ad')